# Milestone 2 lab, notebook 8 of 8: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| **8. The production version** (you are here) | The same logic as a tested program |

The same logic exists as a small program in the `daily_orders/` folder. You already used it in notebook 7. This notebook walks through it.

In [1]:
import inspect

from lab_helpers import LAB_DATABASE, api, lab_warehouse, set_morning, show_doc, table

from daily_orders import extract, load, pipeline, warehouse
from daily_orders.report import format_answer

## Step 1: The same parts, as files

In [2]:
show_doc("LLD.md", "1. Files")

**From `docs/LLD.md`**

> ## 1. Files
> 
> Everything is inside `src/m2_load_only_what_changed/`. Nothing imports from another milestone.
> 
> ```
> docs/                     BRD.md, HLD.md, LLD.md
> notebooks/                the lab: eight notebooks, run in order
>   lab_helpers.py          small helpers the notebooks share
> daily_orders/             production code
>   config.py               addresses, the retry settings
>   extract.py              ask for changed orders; retry with backoff
>   warehouse.py            the Postgres database, its tables and views
>   load.py                 land the rows and move the watermark
>   report.py               read the report
>   pipeline.py             one morning, start to end
>   flow_data.py            builds the numbers shown on the flow page and the recap page
> tests/                    one test file per code file
> milestone_2_flow.html     the whole pipeline, animated
> milestone_2_recap.html    the closing page
> ```
> 
> The warehouse is started by `docker-compose.yml` at the project root.

Here is the retry. Compare it with the `get_page` you wrote in notebook 3.

In [3]:
print(inspect.getsource(extract.get_page))

def get_page(
    client: httpx.Client,
    params: dict,
    extract: Extract,
    sleep: Callable[[float], None] = time.sleep,
) -> dict:
    """Ask for one page. Wait and try again when the API says "not now".

    429 (too many requests): wait as long as the Retry-After header says.
    500 to 504 (the server failed): wait 1, 2, 4, 8 seconds. That is backoff.
    Anything else is our mistake. A retry cannot fix it, so stop.
    """
    for attempt in range(1, MAX_TRIES + 1):
        answer = client.get("/orders", params=params)
        extract.api_calls += 1
        if answer.status_code == 200:
            return answer.json()
        if answer.status_code == 429:
            wait = float(answer.headers.get("Retry-After", 1))
        elif 500 <= answer.status_code <= 504:
            wait = BACKOFF_SECONDS[min(attempt, len(BACKOFF_SECONDS)) - 1]
        else:
            raise ExtractFailed(f"The API answered {answer.status_code}: {answer.text[:200]}")
        if attempt == MAX_TR

One difference matters: `sleep` is passed in. In real use it is `time.sleep`. In a test it is a list's `append`, so the test can check the waits without waiting.

And here is the load. Find the three steps that share one transaction.

In [4]:
print(inspect.getsource(load.land))

def land(
    con: psycopg.Connection,
    orders: list[dict],
    pages: list[int],
    run_id: str,
    run_date: date,
    loaded_at: datetime,
    api_calls: int,
    retries: int,
) -> datetime | None:
    """Append the orders to the landing table, then save the new watermark. One transaction.

    Returns the new watermark.
    """
    before = read_watermark(con)
    try:
        with con.cursor() as cursor:
            cursor.executemany(
                "INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at) VALUES (%s, %s, %s, %s)",
                [(Jsonb(order), run_id, page, loaded_at) for order, page in zip(orders, pages)],
            )
        # Save the largest updated_at we read. Not the time the job ran.
        newest = max((order["updated_at"] for order in orders), default=None)
        after = before
        if newest is not None:
            after = con.execute(
                """
                INSERT INTO ops.watermarks (source, watermark, saved

## Step 2: Run it

A fresh warehouse, then Monday and Tuesday.

In [5]:
lab_warehouse(fresh=True).close()
con = warehouse.connect(LAB_DATABASE)

for day in ["2026-06-08", "2026-06-09"]:
    set_morning(day)
    result = pipeline.run(api, con)

print(f"Run {result.run_id}: asked for orders changed since {result.watermark_before}.")
print(f"Landed {result.rows_landed} rows in {result.api_calls} API calls.\n")
print(format_answer(result.answer))

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
Run 2026-06-09_1: asked for orders changed since 2026-06-08T03:48:53Z.
Landed 29 rows in 2 API calls.

Arden daily orders. Report for the morning of Tue 2026-06-09

  Yesterday   Mon 2026-06-08    25 orders      $591.75
  Day before  Sun 2026-06-07    33 orders      $941.47
  Change                         -8 orders     -$349.72


You can run it without a notebook. From this milestone's folder, in a terminal:

```
uv run python -m daily_orders
```

## Step 3: The pipeline keeps a log

Every run writes one row about itself. This is a first, small piece of the **observe** stage.

In [6]:
table(con, "SELECT run_id, run_date, watermark_before, watermark_after, rows_landed, api_calls, retries FROM ops.load_runs ORDER BY run_id")

,run_id,run_date,watermark_before,watermark_after,rows_landed,api_calls,retries
0,2026-06-08_1,2026-06-08,NaT,2026-06-08 03:48:53+00:00,226,10,0
1,2026-06-09_1,2026-06-09,2026-06-08 03:48:53+00:00,2026-06-09 04:17:31+00:00,29,2,0


In [7]:
con.close()

## Step 4: The tests

The tests start their own private order system. They do need the warehouse running.

In [8]:
!cd .. && uv run pytest -q --color=no 2>&1 | tail -n 2

...............................                                          [100%]
31 passed in 2.58s


In [9]:
show_doc("LLD.md", "7. Checks")

**From `docs/LLD.md`**

> ## 7. Checks
> 
> | Check | Where |
> |---|---|
> | Only changed orders come back, and the boundary order is read again | `tests/test_extract.py` |
> | A 500 is retried with waits of 1, 2, 4 seconds. A 400 is not retried | `tests/test_extract.py` |
> | After 5 failed tries the extract stops | `tests/test_extract.py` |
> | Rows land exactly as received, with load metadata | `tests/test_load.py` |
> | Rows and watermark are saved together or not at all | `tests/test_load.py` |
> | A failed extract lands nothing, and the next run loses nothing | `tests/test_pipeline.py` |
> | The load survives 11 June | `tests/test_pipeline.py` |
> | What is still wrong, measured (section 8) | `tests/test_pipeline.py` |
> | Every number on the pages comes from the code | `tests/test_flow_page.py` |

Some of those tests are unusual: they prove what is **wrong**. A test called `test_still_wrong_a_changed_order_is_counted_twice` pins the exact numbers of the mistake. Milestone 3 will turn each of them around.

## Step 5: What is still wrong

In [10]:
show_doc("LLD.md", "8. What is still wrong")

**From `docs/LLD.md`**

> ## 8. What is still wrong
> 
> Measured with this code, after the runs of 8 and 9 June:
> 
> | | Rows in the landing table | Different orders | In the table twice |
> |---|---|---|---|
> | After two mornings | 255 | 251 | 4 |
> 
> | Day | The truth | The report | Why |
> |---|---|---|---|
> | Fri 06-05 | 37 orders, $971.04 | 38 orders, $985.79 | Order 1130 is counted before and after its refund |
> | Sun 06-07 | 30 orders, $807.11 | 33 orders, $941.47 | Three evening orders changed status the next morning |
> 
> And it gets worse:
> 
> | What you do | What happens |
> |---|---|
> | Run 9 June a second time | 1 more row. Monday shows 26 orders, not 25 |
> | Run 11 June | The API serves page 2 twice. 49 rows land for 37 orders |
> 
> **A changed order is now in the table twice, and the report counts it twice. Run the load two times and it is worse.** Milestone 3 makes the load safe to run twice.

## The words of milestone 2

| Word | In one or two sentences | You saw it in |
|---|---|---|
| **Incremental load** | It reads only rows that are new or changed | Notebook 1 |
| **Incremental key** | The column that says a row changed. Here: `updated_at` | Notebook 1 |
| **Watermark** | The saved value that marks how far the last run read. Also called a high-water mark | Notebook 2 |
| **Pagination** | An API returns rows in pages. You ask again until it says it is done | Notebook 3 |
| **Rate limit** | A cap on how many calls you may make in a period | Notebook 3 |
| **Retry with backoff** | Try a failed call again, and wait longer each time | Notebook 3 |
| **Raw landing table** | The raw copy of what arrived | Notebook 4 |
| **Load metadata** | Columns about the load, not the data: which run, when | Notebook 4 |
| **Append** | Add rows at the end. Never touch the rows already there | Notebook 4 |
| **Pull**, **push** | Who starts the transfer: your job, or the source | Notebook 5 |
| **Polling** | Asking again and again: what is new? | Notebook 5 |
| **Change data capture (CDC)** | Capturing each change to a table. A watermark load is query-based CDC | Notebook 6 |

The project's [`GLOSSARY.md`](../../../GLOSSARY.md) keeps every word, with an interview line for each.

## What you now know

- The program reads only what changed, waits when the API says no, and lands everything raw.
- Rows, watermark and run log are saved together or not at all.
- Its tests pin down both what works and what is still wrong.

**You have finished milestone 2.** Open [`milestone_2_recap.html`](../milestone_2_recap.html) in a browser for a look back. Then open [`milestone_2_flow.html`](../milestone_2_flow.html) again.

> **Still wrong:** a changed order is now in the table twice, and the report counts it twice. Run the load two times and it is worse. Milestone 3 makes the load safe to run twice.